# S3 - directions + probes + control_directions + projections

Target this session at **240-270 min** wall clock; hard boundary **300 min**. Do not start a stage/condition that the calibrated projection says cannot finish (analysis_plan.md §7).

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone and pin the exact commit

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR = '/content/dpo-safety-representations'
BRANCH = 'main'
PINNED_COMMIT = '37c22f4fa0d8a1098017f5518cdeb0b7ad4cf5dd'   # exact commit this session runs against

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", PINNED_COMMIT], check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert commit == PINNED_COMMIT, f"wrong commit: {commit} != {PINNED_COMMIT}"
print("checked out", commit)

## 3. Install dependencies, check GPU

In [ ]:
!pip -q install -r requirements.txt
# Colab preinstalls torchao; it breaks transformers on quantized/8-bit
# loads (see 04b). Matches colab_unified_{analysis,training}.ipynb.
!pip uninstall -y torchao || true
!nvidia-smi

## 4. Persistent storage (results/ + HF cache bound to Drive)

In [ ]:
# Bind results/ + the HF weight cache to a persistent Drive folder so this
# session's work survives a Colab disconnect and a fresh VM resumes it. One
# line - the logic + tests live in src/colab_persist.py. Idempotent, and safe
# even if you have already done work on the ephemeral results/ (it merges that
# into Drive first). Override the Drive root with the DPO_DRIVE_ROOT env var;
# pass persist_hf_cache=False to keep the ~5 GB HF cache off Drive.
from src.colab_persist import bind, status_line
info = bind()                     # or: bind(persist_hf_cache=False)
print(status_line(info))
!python -m src.analysis.v2_pipeline status

## 5. Directions (force past stale 370-era outputs)

In [ ]:
!python -m src.analysis.v2_pipeline direction --force --stages M0 M1 M2 M3 M3_direct M1_alt M2_alt M3_alt M3_direct_alt

## 6. Probes (fixed FINAL_LAYER headline; no C/D selection)

In [ ]:
!python -m src.analysis.v2_pipeline probes --stages M0 M1 M2 M3 M3_direct M1_alt M2_alt M3_alt M3_direct_alt

## 7. Control directions (seeded r, calibration-RMS gamma, d_AB)

In [ ]:
!python -m src.analysis.control_directions

## 8. Canonical `_final` per-prompt + fixed-reference projections

In [ ]:
!python -m src.analysis.representation_projections

## 9. Decide `ablated_AB` by calibrated session fit

In [ ]:
from src.analysis.intervention_conditions import plan_causal_conditions
print(plan_causal_conditions('M3', per_condition_minutes=30, budget_minutes=270, requested=['baseline','ablated_AD','ablated_random','ablated_AB']).to_json())